In [6]:
import numpy as np
import pandas as pd
from skimage import io, measure, color, filters
from skimage.feature import graycomatrix, graycoprops
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree # --- NEW --- For relational features


In [2]:
# --- Load Data ---
import os
im_path = r"C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\image and seg data"
file_name= os.listdir(im_path)
print(file_name)  


mask_path = os.path.join(im_path, 'C1-C30000_seg.npy')
image_path = os.path.join(im_path, 'C1-C30000.png')

# Load the image (convert to grayscale if it's RGB)
image = io.imread(image_path)
if image.ndim == 3:
    image = color.rgb2gray(image)
# Ensure image is in a standard format (e.g., 8-bit or 16-bit unsigned integer)
image = (image / np.max(image) * 255).astype(np.uint8)


seg_data = np.load(mask_path, allow_pickle=True).item()
# The labeled mask where each nucleus has a unique integer ID
labeled_mask = seg_data['masks'] 

print(f"Loaded image of shape: {image.shape}")
print(f"Loaded labeled mask of shape: {labeled_mask.shape}")

['C1-C30000.png', 'C1-C30000.png_features.csv', 'C1-C30000_seg.npy', 'C1-C30001.png', 'C1-C30001_seg.npy', 'C1-C30002.png', 'C1-C30002_seg.npy', 'C1-C30003.png', 'C1-C30003_seg.npy', 'C1-C30004.png', 'C1-C30004_seg.npy', 'C1-C30005.png', 'C1-C30005_seg.npy', 'C1-C40000.png', 'C1-C40000_seg.npy', 'C1-C40001.png', 'C1-C40001_seg.npy', 'C1-C40002.png', 'C1-C40002_seg.npy', 'C1-C40003.png', 'C1-C40003_seg.npy', 'C1-C50000.png', 'C1-C50000_seg.npy', 'C1-C50001.png', 'C1-C50001_seg.npy', 'C1-C50002.png', 'C1-C50002_seg.npy', 'C1-C50003.png', 'C1-C50003_seg.npy', 'C1-D10000.png', 'C1-D10000_seg.npy', 'C1-D10001.png', 'C1-D10001_seg.npy', 'C1-D10002.png', 'C1-D10002_seg.npy', 'C1-D10003.png', 'C1-D10003_seg.npy', 'C2-C30000.png', 'C2-C30000_seg.npy', 'C2-C30001.png', 'C2-C30001_seg.npy', 'C2-C30002.png', 'C2-C30002_seg.npy', 'C2-C30003.png', 'C2-C30003_seg.npy', 'C2-C30004.png', 'C2-C30004_seg.npy', 'C2-C30005.png', 'C2-C30005_seg.npy', 'C2-C40000.png', 'C2-C40000_seg.npy', 'C2-C40001.png', 'C

In [7]:

# --- 2. Define All Feature Calculation Functions ---

def calculate_circularity(region):
    if region.perimeter == 0: return 0
    return (4 * np.pi * region.area) / (region.perimeter ** 2)

def get_glcm_features(region):
    patch = region.intensity_image
    if patch.size == 0:
        return {'glcm_contrast': 0, 'glcm_homogeneity': 0, 'glcm_energy': 0, 'glcm_correlation': 0}
    glcm = graycomatrix(patch, distances=[1], angles=[0], levels=256, symmetric=True, normed=False)
    glcm[0, :, :, :] = 0
    glcm[:, 0, :, :] = 0
    total = np.sum(glcm)
    if total == 0:
        return {'glcm_contrast': 0, 'glcm_homogeneity': 0, 'glcm_energy': 0, 'glcm_correlation': 0}
    glcm_norm = glcm / total
    contrast = graycoprops(glcm_norm, 'contrast')[0, 0]
    homogeneity = graycoprops(glcm_norm, 'homogeneity')[0, 0]
    energy = graycoprops(glcm_norm, 'energy')[0, 0]
    correlation = graycoprops(glcm_norm, 'correlation')[0, 0]
    return {'glcm_contrast': contrast, 'glcm_homogeneity': homogeneity, 'glcm_energy': energy, 'glcm_correlation': correlation}

# --- NEW --- Function to calculate sub-nuclear (chromocenter) features
def get_subnuclear_features(region):
    """Analyzes bright spots (chromocenters) within the nucleus."""
    patch = region.intensity_image
    
    # Check for empty patch or uniform patch which would cause thresholding to fail
    if patch.size == 0 or np.all(patch == patch[0,0]):
        return {'n_chromocenters': 0, 'mean_chromocenter_area': 0, 'chromocenter_area_fraction': 0}

    # Apply Otsu's threshold to find bright spots (heterochromatin)
    try:
        thresh = filters.threshold_otsu(patch[patch > 0]) # Threshold only foreground pixels
        chromocenter_mask = patch > thresh
    except ValueError: # Handle cases where Otsu's method fails
        return {'n_chromocenters': 0, 'mean_chromocenter_area': 0, 'chromocenter_area_fraction': 0}
        
    # Label the detected spots
    labeled_chromocenters = measure.label(chromocenter_mask)
    n_chromocenters = np.max(labeled_chromocenters)
    
    if n_chromocenters == 0:
        return {'n_chromocenters': 0, 'mean_chromocenter_area': 0, 'chromocenter_area_fraction': 0}
        
    # Calculate features of the spots
    chromocenter_props = measure.regionprops(labeled_chromocenters)
    total_chromocenter_area = np.sum([prop.area for prop in chromocenter_props])
    mean_chromocenter_area = total_chromocenter_area / n_chromocenters
    chromocenter_area_fraction = total_chromocenter_area / region.area
    
    return {
        'n_chromocenters': n_chromocenters, 
        'mean_chromocenter_area': mean_chromocenter_area, 
        'chromocenter_area_fraction': chromocenter_area_fraction
    }


In [8]:

# --- 3. Extract Features for All Cells ---

# Basic properties to extract directly
properties_basic = [
    'label', 'area', 'perimeter', 'solidity', 'extent',
    'mean_intensity', 'max_intensity', 'min_intensity',
    'eccentricity', 'major_axis_length', 'minor_axis_length', 'orientation'
]

# This is the main list that will hold all feature dictionaries
all_features = []
props = measure.regionprops(labeled_mask, intensity_image=image)

# --- NEW --- Pre-computation for relational features
# Get all cell centroids to build the neighbor-finding tree
all_centroids = [p.centroid for p in props]
if len(all_centroids) > 1:
    kdtree = cKDTree(all_centroids)
else:
    kdtree = None # Cannot compute relational features if there's only one cell


# Main loop through each detected cell
print(f"Extracting features for {len(props)} cells...")
for region in props:
    # --- Basic and GLCM Features ---
    features = {prop: getattr(region, prop) for prop in properties_basic}
    features['circularity'] = calculate_circularity(region)
    features['intensity_std'] = np.std(region.intensity_image[region.image])
    features.update(get_glcm_features(region))
    
    # --- NEW --- Add Cell Height (Y-centroid) ---
    features['cell_height_y'] = region.centroid[0]

    # --- NEW --- Add Relational Features ---
    if kdtree:
        # Find 6 nearest neighbors (the first one will be the cell itself)
        distances, _ = kdtree.query(region.centroid, k=6)
        # Calculate the mean distance to the 5 actual neighbors
        features['mean_dist_to_5_neighbors'] = np.mean(distances[1:])
    else:
        features['mean_dist_to_5_neighbors'] = 0 # Default value if no neighbors

    # --- NEW --- Add Sub-nuclear Features ---
    features.update(get_subnuclear_features(region))

    all_features.append(features)








Extracting features for 161 cells...


In [9]:

# --- 4. Create and Save DataFrame ---
df = pd.DataFrame(all_features)

# Add unique identifiers
image_basename = image_path.split('/')[-1].replace('.jpg', '')
df['unique_id'] = f"{image_basename}_cell_" + df['label'].astype(str)
df.set_index('unique_id', inplace=True)

# Save the comprehensive DataFrame
output_filename = f'c1c3000_features_extended.csv'
df.to_csv(output_filename)

print(f"\nSuccessfully extracted extended features and saved to '{output_filename}'")
df.head()


Successfully extracted extended features and saved to 'c1c3000_features_extended.csv'


,label,area,perimeter,solidity,extent,mean_intensity,max_intensity,min_intensity,eccentricity,major_axis_length,...,intensity_std,glcm_contrast,glcm_homogeneity,glcm_energy,glcm_correlation,cell_height_y,mean_dist_to_5_neighbors,n_chromocenters,mean_chromocenter_area,chromocenter_area_fraction
unique_id,,,,,,,,,,,,,,,,,,,,,
C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\image and seg data\C1-C30000.png_cell_1,1,762.0,105.941125,0.963338,0.814103,119.308399,255.0,12.0,0.706185,37.446909,...,57.862840,1271.915761,0.041787,0.028273,0.806814,11.828084,67.536428,6,53.666667,0.422572
C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\image and seg data\C1-C30000.png_cell_2,2,1191.0,126.083261,0.981054,0.847084,106.036944,255.0,9.0,0.254669,39.663566,...,50.464978,1125.891587,0.039730,0.023207,0.774130,18.047019,59.976544,16,28.750000,0.386230
C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\image and seg data\C1-C30000.png_cell_3,3,1177.0,128.325902,0.974338,0.797425,88.415463,206.0,8.0,0.579887,43.033066,...,34.960882,937.094190,0.043736,0.024427,0.604410,19.292268,60.765543,10,55.700000,0.473237
C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\image and seg data\C1-C30000.png_cell_4,4,697.0,103.840620,0.969402,0.757609,145.647059,255.0,18.0,0.822307,40.011962,...,54.909253,1337.267062,0.047559,0.030873,0.769421,9.868006,71.669982,11,28.272727,0.446198
C:\Users\shiva\OneDrive\Desktop\MASK-R-CNN-Vision-Biology-Lab\image and seg data\C1-C30000.png_cell_5,5,395.0,80.526912,0.940476,0.707885,73.962025,167.0,9.0,0.772653,29.112273,...,29.522719,725.442971,0.069568,0.041007,0.571550,7.769620,53.337739,8,21.000000,0.425316
